Shahlo Solieva

Updated 10/14/2026

Use this notebook to align by sequence and save data into csv files 
ci = 95% bootstrapped percent intervals. Bootstraps with replacement. 

In [6]:
import numpy as np
import mdtraj as md
import pandas as pd

BASE_PATH = "/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/RMSF"
CLUSTER_BASE = "/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/clustering/radius_15_variable/data"

# Helper functions

In [7]:
def get_residues(sys_path, vh_start, vh_end, vl_start, vl_end):
    """
    Return heavy and light chain residue names from variable regions.
    """
    pdb_path = f"{CLUSTER_BASE}/{sys_path}/prot_masses.pdb"
    pdb = md.load(pdb_path)
    ca_atoms = pdb.topology.select(
        f'chainid 0 and residue {vh_start} to {vh_end} and name CA '
        f'or chainid 1 and residue {vl_start} to {vl_end} and name CA')
    residues = [pdb.topology.atom(i).residue for i in ca_atoms]
    hc_residues = residues[:vh_end]
    lc_residues = residues[vh_end:]
    return hc_residues, lc_residues

In [8]:
def load_rmsf_and_split(system_name, vh_end, variable_name):
    """
    Load RMSF data for given system

    """
    path = f"{BASE_PATH}/data/apo/{system_name}/VHC_VLC_CA_RMSF_{variable_name}.npy"
    rmsf_data = np.load(path, allow_pickle=True)
    hc = rmsf_data[:vh_end] * 10
    lc = rmsf_data[vh_end:] * 10
    return hc, lc

def calculate_95CI_and_split(system_name, vh_end, variable_name):
    """
    Load bootstrapped (n=1000) RMSF for given system, split into hc and lc data. 
    """
    DATA_PATH = f"{BASE_PATH}/data/apo/{system_name}/n1000_bootstraps_VHC_VLC_CA_RMSF_avgs_{variable_name}.npy"
    rmsfs_all_average = np.load(DATA_PATH)*10
    ci_low  = np.percentile(rmsfs_all_average, 2.5,  axis=0) 
    ci_high = np.percentile(rmsfs_all_average, 97.5, axis=0)  
    ci_hc   = [ci_low[:vh_end], ci_high[:vh_end]]  # ci low for HC, ci high for HC
    ci_lc   = [ci_low[vh_end:], ci_high[vh_end:]] 
    return ci_hc, ci_lc 

In [9]:
def insert_gaps(array, indices):
    """Insert None values into array at given indices."""
    
    result = array.copy()
    for idx in sorted(indices):
        result = np.insert(result, idx, None)
    return result

In [10]:
def load_residues_and_rmsf(system_name, variable_name, vh_start, vh_end, vl_start, vl_end):
    """
    Load heavy and light chain residues and rmsf values for a system.
    Returns:
        hc_resi: list of heavy chain residue names
        lc_resi: list of light chain residue names
        hc_rmsf: numpy array of heavy chain RMSF values (Å)
        lc_rmsf: numpy array of light chain RMSF values (Å)
    """
    # Get residues
    hc_resi, lc_resi = get_residues(f"apo/{system_name}", vh_start, vh_end, vl_start, vl_end)

    # Load RMSF and Split into HC and LC RMSF
    hc_rmsf, lc_rmsf = load_rmsf_and_split(system_name, vh_end, variable_name)
    ci_hc, ci_lc = calculate_95CI_and_split(system_name, vh_end, variable_name) 
    return hc_resi, lc_resi, hc_rmsf, lc_rmsf, ci_hc, ci_lc

In [11]:
def apply_alignment_gaps(resi_list, rmsf_array, gap_positions, err=None):
    """
    Insert None values at specified positions to align sequence and rmsf.
    Args:
        resi_list (list or array): residue names
        rmsf_array (array): rmsf values
        gap_positions (list of int): indices where None should be inserted
    Returns:
        resi_aligned (np.ndarray): residues with gaps inserted
        rmsf_aligned (np.ndarray): rmsf values with gaps inserted
    """
    resi_aligned = np.array(resi_list)
    rmsf_aligned = rmsf_array.copy()
    
    if err is not None:
        low_ci_aligned  = err[0].copy()
        high_ci_aligned = err[1].copy()
        #rmsf_std_aligned=err.copy()
        for idx in gap_positions:
            resi_aligned = np.insert(resi_aligned, idx, None)
            rmsf_aligned = np.insert(rmsf_aligned, idx, None)
            low_ci_aligned = np.insert(low_ci_aligned, idx, None)
            high_ci_aligned = np.insert(high_ci_aligned, idx, None)
        return resi_aligned, rmsf_aligned, [low_ci_aligned, high_ci_aligned]
    else:
        for idx in gap_positions:
            resi_aligned = np.insert(resi_aligned, idx, None)
            rmsf_aligned = np.insert(rmsf_aligned, idx, None)
        return resi_aligned, rmsf_aligned

In [12]:
variable_name = 'lagtime50' #'radius18' # 'lagtime300'

# CAP256

In [13]:
print("Importing rmsf data for naive antibody")
naive_name = "CAP256_naive_VC"
naive_ranges = (1, 144, 1, 111)
naive_hc_resi, naive_lc_resi, naive_hc_rmsf, naive_lc_rmsf, naive_hc_rmsf_ci, naive_lc_rmsf_ci = \
    load_residues_and_rmsf(naive_name, variable_name, *naive_ranges)

print("Importing rmsf data for mature antibody")
mature_name = "CAP256_mature_VC"
mature_ranges = (1, 145, 1, 111)
mature_hc_resi, mature_lc_resi, mature_hc_rmsf, mature_lc_rmsf, mature_hc_rmsf_ci, mature_lc_rmsf_ci = \
    load_residues_and_rmsf(mature_name, variable_name, *mature_ranges)

Importing rmsf data for naive antibody


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/RMSF/data/apo/CAP256_naive_VC/n1000_bootstraps_VHC_VLC_CA_RMSF_avgs_lagtime50.npy'

In [126]:
# Sequence alignment gaps for heavy chain
naive_gap_positions = [118, 118, 122, 128, 133]
mature_gap_positions = [115, 115, 126, 135]

naive_resi_fixed, naive_rmsf_fixed, naive_rmsf_ci_fixed = apply_alignment_gaps(
    naive_hc_resi, naive_hc_rmsf, naive_gap_positions, naive_hc_rmsf_ci)
mature_resi_fixed, mature_rmsf_fixed, mature_rmsf_ci_fixed = apply_alignment_gaps(
    mature_hc_resi, mature_hc_rmsf, mature_gap_positions, mature_hc_rmsf_ci)

# Save light chain alignment
df = pd.DataFrame({
    "naive_HC_resi":    naive_resi_fixed,
    "naive_HC_rmsf":    naive_rmsf_fixed,
    "naive_HC_rmsf_ci_low":naive_rmsf_ci_fixed[0],
    "naive_HC_rmsf_ci_high":naive_rmsf_ci_fixed[1],
    "mature_HC_resi":   mature_resi_fixed,
    "mature_HC_rmsf":   mature_rmsf_fixed,
    "mature_HC_rmsf_ci_low":mature_rmsf_ci_fixed[0],
    "mature_HC_rmsf_ci_high":mature_rmsf_ci_fixed[1]})
output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/CAP256_mature_aligned_to_naive_HC_{variable_name}.csv"
df.to_csv(output, index=False)

In [127]:
# Sequence alignment gaps for light chain
# none

# Save light chain alignment
df = pd.DataFrame({
    "naive_LC_resi": naive_lc_resi,
    "naive_LC_rmsf": naive_lc_rmsf,
    "naive_LC_rmsf_ci_low": naive_lc_rmsf_ci[0],
    "naive_LC_rmsf_ci_high":naive_lc_rmsf_ci[1],
    "mature_LC_resi": mature_lc_resi,
    "mature_LC_rmsf": mature_lc_rmsf,
    "mature_LC_rmsf_ci_low": mature_lc_rmsf_ci[0],
    "mature_LC_rmsf_ci_high":mature_lc_rmsf_ci[1]})
output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/CAP256_mature_aligned_to_naive_LC_{variable_name}.csv"
df.to_csv(output, index=False)

# CH103

In [8]:
print("Importing rmsf data for naive antibody")
naive_name = "CH103_naive_VC"
naive_ranges = (1, 121, 1, 108)
naive_hc_resi, naive_lc_resi, naive_hc_rmsf, naive_lc_rmsf, naive_hc_rmsf_ci, naive_lc_rmsf_ci = \
    load_residues_and_rmsf(naive_name, variable_name, *naive_ranges)

print("Importing rmsf data for mature antibody")
mature_name = "CH103_mature_VC"
mature_ranges = (1, 121, 1, 105)
mature_hc_resi, mature_lc_resi, mature_hc_rmsf, mature_lc_rmsf, mature_hc_rmsf_ci, mature_lc_rmsf_ci = \
    load_residues_and_rmsf(mature_name, variable_name, *mature_ranges)

Importing rmsf data for naive antibody
Importing rmsf data for mature antibody


In [9]:
len(mature_lc_rmsf_ci[0])

105

In [10]:
# Sequence alignment gaps for heavy chain
# none

# Save heavy chain alignment
df = pd.DataFrame({
    "naive_HC_resi": naive_hc_resi,
    "naive_HC_rmsf": naive_hc_rmsf,
    "naive_HC_rmsf_ci_low": naive_hc_rmsf_ci[0],
    "naive_HC_rmsf_ci_high":naive_hc_rmsf_ci[1],
    "mature_HC_resi": mature_hc_resi,
    "mature_HC_rmsf": mature_hc_rmsf,
    "mature_HC_rmsf_ci_low": mature_hc_rmsf_ci[0],
    "mature_HC_rmsf_ci_high":mature_hc_rmsf_ci[1]}) 

output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/CH103_mature_aligned_to_naive_HC_{variable_name}.csv"
df.to_csv(output, index=False)

In [11]:
# Sequence alignment gaps for light chain
naive_gap_positions = []
mature_gap_positions = [27,29,29]

naive_resi_fixed, naive_rmsf_fixed, naive_rmsf_ci_fixed = apply_alignment_gaps(
    naive_lc_resi, naive_lc_rmsf, naive_gap_positions, naive_lc_rmsf_ci)
mature_resi_fixed, mature_rmsf_fixed, mature_rmsf_ci_fixed = apply_alignment_gaps(
    mature_lc_resi, mature_lc_rmsf, mature_gap_positions, mature_lc_rmsf_ci)
print(len(naive_rmsf_ci_fixed[0]), len(mature_rmsf_ci_fixed[1]))
# Save heavy chain alignment
df = pd.DataFrame({
    "naive_LC_resi":    naive_resi_fixed,
    "naive_LC_rmsf":    naive_rmsf_fixed,
    "naive_LC_rmsf_ci_low":naive_rmsf_ci_fixed[0],
    "naive_LC_rmsf_ci_high":naive_rmsf_ci_fixed[1],
    "mature_LC_resi":   mature_resi_fixed,
    "mature_LC_rmsf":   mature_rmsf_fixed,
    "mature_LC_rmsf_ci_low":mature_rmsf_ci_fixed[0],
    "mature_LC_rmsf_ci_high":mature_rmsf_ci_fixed[1]})
output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/CH103_mature_aligned_to_naive_LC_{variable_name}.csv"
df.to_csv(output, index=False)

108 108


# EPTC112

In [132]:
print("Importing rmsf data for naive antibody")
naive_name = "EPTC112_naive_VC"
naive_ranges = (1, 121, 1, 110)
naive_hc_resi, naive_lc_resi, naive_hc_rmsf, naive_lc_rmsf, naive_hc_rmsf_ci, naive_lc_rmsf_ci = \
    load_residues_and_rmsf(naive_name, variable_name, *naive_ranges)

print("Importing rmsf data for mature antibody")
mature_name = "EPTC112_mature_VC"
mature_ranges = (1, 121, 1, 110)
mature_hc_resi, mature_lc_resi, mature_hc_rmsf, mature_lc_rmsf, mature_hc_rmsf_ci, mature_lc_rmsf_ci = \
    load_residues_and_rmsf(mature_name, variable_name, *mature_ranges)

Importing rmsf data for naive antibody
Importing rmsf data for mature antibody


In [133]:
# Sequence alignment gaps for heavy chain
# none

# Save heavy chain alignment
df = pd.DataFrame({
    "naive_HC_resi": naive_hc_resi,
    "naive_HC_rmsf": naive_hc_rmsf,
    "naive_HC_rmsf_ci_low": naive_hc_rmsf_ci[0],
    "naive_HC_rmsf_ci_high":naive_hc_rmsf_ci[1],
    "mature_HC_resi": mature_hc_resi,
    "mature_HC_rmsf": mature_hc_rmsf,
    "mature_HC_rmsf_ci_low": mature_hc_rmsf_ci[0],
    "mature_HC_rmsf_ci_high":mature_hc_rmsf_ci[1]})
output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/EPTC112_mature_aligned_to_naive_HC_{variable_name}.csv"
df.to_csv(output, index=False)

In [134]:
# Sequence alignment gaps for light chain
# none

# Save light chain alignment
df = pd.DataFrame({
    "naive_LC_resi": naive_lc_resi,
    "naive_LC_rmsf": naive_lc_rmsf,
    "naive_LC_rmsf_ci_low": naive_lc_rmsf_ci[0],
    "naive_LC_rmsf_ci_high":naive_lc_rmsf_ci[1],
    "mature_LC_resi": mature_lc_resi,
    "mature_LC_rmsf": mature_lc_rmsf,
    "mature_LC_rmsf_ci_low": mature_lc_rmsf_ci[0],
    "mature_LC_rmsf_ci_high":mature_lc_rmsf_ci[1]})
output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/EPTC112_mature_aligned_to_naive_LC_{variable_name}.csv"
df.to_csv(output, index=False)

# CH65

In [135]:
print("Importing rmsf data for naive antibody")
naive_name = "CH65_naive_VC"
naive_ranges = (1, 126, 1, 108)
naive_hc_resi, naive_lc_resi, naive_hc_rmsf, naive_lc_rmsf, naive_hc_rmsf_ci, naive_lc_rmsf_ci = \
    load_residues_and_rmsf(naive_name, variable_name, *naive_ranges)

print("Importing rmsf data for mature antibody")
mature_name = "CH65_mature_VC"
mature_ranges = (1, 126, 1, 108)
mature_hc_resi, mature_lc_resi, mature_hc_rmsf, mature_lc_rmsf, mature_hc_rmsf_ci, mature_lc_rmsf_ci = \
    load_residues_and_rmsf(mature_name, variable_name, *mature_ranges)

# Sequence alignment gaps for heavy chain
# none

# Save heavy chain alignment
df = pd.DataFrame({
    "naive_HC_resi": naive_hc_resi,
    "naive_HC_rmsf": naive_hc_rmsf,
    "naive_HC_rmsf_ci_low": naive_hc_rmsf_ci[0],
    "naive_HC_rmsf_ci_high":naive_hc_rmsf_ci[1],
    "mature_HC_resi": mature_hc_resi,
    "mature_HC_rmsf": mature_hc_rmsf,
    "mature_HC_rmsf_ci_low": mature_hc_rmsf_ci[0],
    "mature_HC_rmsf_ci_high":mature_hc_rmsf_ci[1]})
output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/CH65_mature_aligned_to_naive_HC_{variable_name}.csv"
df.to_csv(output, index=False)

# Sequence alignment gaps for light chain
# none

# Save light chain alignment
df = pd.DataFrame({
    "naive_LC_resi": naive_lc_resi,
    "naive_LC_rmsf": naive_lc_rmsf,
    "naive_LC_rmsf_ci_low": naive_lc_rmsf_ci[0],
    "naive_LC_rmsf_ci_high":naive_lc_rmsf_ci[1],
    "mature_LC_resi": mature_lc_resi,
    "mature_LC_rmsf": mature_lc_rmsf,
    "mature_LC_rmsf_ci_low": mature_lc_rmsf_ci[0],
    "mature_LC_rmsf_ci_high":mature_lc_rmsf_ci[1]})
output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/CH65_mature_aligned_to_naive_LC_{variable_name}.csv"
df.to_csv(output, index=False)

Importing rmsf data for naive antibody
Importing rmsf data for mature antibody


# PGT121 and PGT124

## PGT121

In [136]:
print("Importing rmsf data for naive antibody")
naive_name = "PGT121_naive_kappa_VC"
naive_ranges = (1, 132, 1, 109)
naive_hc_resi, naive_lc_resi, naive_hc_rmsf, naive_lc_rmsf, naive_hc_rmsf_ci, naive_lc_rmsf_ci = \
    load_residues_and_rmsf(naive_name, variable_name, *naive_ranges)

print("Importing rmsf data for mature antibody")
mature_name = "PGT121_mature_kappa_VC"
mature_ranges = (1, 132, 1, 105)
mature_hc_resi, mature_lc_resi, mature_hc_rmsf, mature_lc_rmsf, mature_hc_rmsf_ci, mature_lc_rmsf_ci = \
    load_residues_and_rmsf(mature_name, variable_name, *mature_ranges)

# Sequence alignment gaps for heavy chain
# none

# Save heavy chain alignment
df = pd.DataFrame({
    "naive_HC_resi": naive_hc_resi,
    "naive_HC_rmsf": naive_hc_rmsf,
    "naive_HC_rmsf_ci_low": naive_hc_rmsf_ci[0],
    "naive_HC_rmsf_ci_high":naive_hc_rmsf_ci[1],
    "mature_HC_resi": mature_hc_resi,
    "mature_HC_rmsf": mature_hc_rmsf,
    "mature_HC_rmsf_ci_low": mature_hc_rmsf_ci[0],
    "mature_HC_rmsf_ci_high":mature_hc_rmsf_ci[1]})
output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/PGT121_mature_aligned_to_naive_HC_{variable_name}.csv"
df.to_csv(output, index=False)

Importing rmsf data for naive antibody
Importing rmsf data for mature antibody


In [137]:
# Sequence alignment gaps for light chain
naive_gap_positions = [9, 65, 68, 68, 101, 101]
mature_gap_positions= [0,0,0,0,0,0,0,0, 98, 98]

naive_resi_fixed, naive_rmsf_fixed, naive_rmsf_ci_fixed = apply_alignment_gaps(
    naive_lc_resi, naive_lc_rmsf, naive_gap_positions, naive_lc_rmsf_ci)
mature_resi_fixed, mature_rmsf_fixed, mature_rmsf_ci_fixed = apply_alignment_gaps(
    mature_lc_resi, mature_lc_rmsf, mature_gap_positions, mature_lc_rmsf_ci)

# Save light chain alignment
df = pd.DataFrame({
    "naive_LC_resi":    naive_resi_fixed,
    "naive_LC_rmsf":    naive_rmsf_fixed,
    "naive_LC_rmsf_ci_low":naive_rmsf_ci_fixed[0],
    "naive_LC_rmsf_ci_high":naive_rmsf_ci_fixed[1],
    "mature_LC_resi":   mature_resi_fixed,
    "mature_LC_rmsf":   mature_rmsf_fixed,
    "mature_LC_rmsf_ci_low":mature_rmsf_ci_fixed[0],
    "mature_LC_rmsf_ci_high":mature_rmsf_ci_fixed[1]})
output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/PGT121_mature_aligned_to_naive_LC_{variable_name}.csv"
df.to_csv(output, index=False)

# 1-18

In [138]:
print("Importing rmsf data for naive antibody")
naive_name = "Ab118_naive_VC"
naive_ranges = (1, 125, 1, 108)
naive_hc_resi, naive_lc_resi, naive_hc_rmsf, naive_lc_rmsf, naive_hc_rmsf_ci, naive_lc_rmsf_ci = \
    load_residues_and_rmsf(naive_name, variable_name, *naive_ranges)

print("Importing rmsf data for mature antibody")
mature_name = "Ab118_mature_VC"
mature_ranges = (1, 131, 1, 108)
mature_hc_resi, mature_lc_resi, mature_hc_rmsf, mature_lc_rmsf, mature_hc_rmsf_ci, mature_lc_rmsf_ci = \
    load_residues_and_rmsf(mature_name, variable_name, *mature_ranges)

# Sequence alignment gaps for light chain
# none

# Save light chain alignment
df = pd.DataFrame({
    "naive_LC_resi": naive_lc_resi,
    "naive_LC_rmsf": naive_lc_rmsf,
    "naive_LC_rmsf_ci_low": naive_lc_rmsf_ci[0],
    "naive_LC_rmsf_ci_high":naive_lc_rmsf_ci[1],
    "mature_LC_resi": mature_lc_resi,
    "mature_LC_rmsf": mature_lc_rmsf,
    "mature_LC_rmsf_ci_low": mature_lc_rmsf_ci[0],
    "mature_LC_rmsf_ci_high":mature_lc_rmsf_ci[1]})
output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/Ab118_mature_aligned_to_naive_LC_{variable_name}.csv"
df.to_csv(output, index=False)

Importing rmsf data for naive antibody
Importing rmsf data for mature antibody


In [139]:
# Sequence alignment gaps for heavy chain
naive_gap_positions = [26,26,29,29,29,29]
mature_gap_positions= []

naive_resi_fixed, naive_rmsf_fixed, naive_rmsf_ci_fixed = apply_alignment_gaps(
    naive_hc_resi, naive_hc_rmsf, naive_gap_positions, naive_hc_rmsf_ci)
mature_resi_fixed, mature_rmsf_fixed, mature_rmsf_ci_fixed = apply_alignment_gaps(
    mature_hc_resi, mature_hc_rmsf, mature_gap_positions, mature_hc_rmsf_ci)

# Save light chain alignment
df = pd.DataFrame({
    "naive_HC_resi":    naive_resi_fixed,
    "naive_HC_rmsf":    naive_rmsf_fixed,
    "naive_HC_rmsf_ci_low":naive_rmsf_ci_fixed[0],
    "naive_HC_rmsf_ci_high":naive_rmsf_ci_fixed[1],
    "mature_HC_resi":   mature_resi_fixed,
    "mature_HC_rmsf":   mature_rmsf_fixed,
    "mature_HC_rmsf_ci_low":mature_rmsf_ci_fixed[0],
    "mature_HC_rmsf_ci_high":mature_rmsf_ci_fixed[1]})
output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/Ab118_mature_aligned_to_naive_HC_{variable_name}.csv"
df.to_csv(output, index=False)

# 3F8

In [140]:
print("Importing rmsf data for naive antibody")
naive_name = "Ab3F8_naive_VC"
naive_ranges = (1, 119, 1, 101)
naive_hc_resi, naive_lc_resi, naive_hc_rmsf, naive_lc_rmsf, naive_hc_rmsf_ci, naive_lc_rmsf_ci = \
    load_residues_and_rmsf(naive_name, variable_name, *naive_ranges)

print("Importing rmsf data for mature antibody")
mature_name = "Ab3F8_mature_VC"
mature_ranges = (1, 119, 1, 101)
mature_hc_resi, mature_lc_resi, mature_hc_rmsf, mature_lc_rmsf, mature_hc_rmsf_ci, mature_lc_rmsf_ci = \
    load_residues_and_rmsf(mature_name, variable_name, *mature_ranges)

# Sequence alignment gaps for heavy chain
# none

# Save heavy chain alignment
df = pd.DataFrame({
    "naive_HC_resi": naive_hc_resi,
    "naive_HC_rmsf": naive_hc_rmsf,
    "naive_HC_rmsf_ci_low": naive_hc_rmsf_ci[0],
    "naive_HC_rmsf_ci_high":naive_hc_rmsf_ci[1],
    "mature_HC_resi": mature_hc_resi,
    "mature_HC_rmsf": mature_hc_rmsf,
    "mature_HC_rmsf_ci_low": mature_hc_rmsf_ci[0],
    "mature_HC_rmsf_ci_high":mature_hc_rmsf_ci[1]})
output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/Ab3F8_mature_aligned_to_naive_HC_{variable_name}.csv"
df.to_csv(output, index=False)

# Sequence alignment gaps for light chain
# none

# Save light chain alignment
df = pd.DataFrame({
    "naive_LC_resi": naive_lc_resi,
    "naive_LC_rmsf": naive_lc_rmsf,
    "naive_LC_rmsf_ci_low": naive_lc_rmsf_ci[0],
    "naive_LC_rmsf_ci_high":naive_lc_rmsf_ci[1],
    "mature_LC_resi": mature_lc_resi,
    "mature_LC_rmsf": mature_lc_rmsf,
    "mature_LC_rmsf_ci_low": mature_lc_rmsf_ci[0],
    "mature_LC_rmsf_ci_high":mature_lc_rmsf_ci[1]})
output = f"{BASE_PATH}/data/seq_alignments/naive_aligned/Ab3F8_mature_aligned_to_naive_LC_{variable_name}.csv"
df.to_csv(output, index=False)

Importing rmsf data for naive antibody
Importing rmsf data for mature antibody
